# Dual number demo in ForwardDiff

Doing second and first derivatives at the same time.  The tags are needed to avoid perturbation confusion.

Analytical Functions

In [68]:
# Define f, g=df, and H=ddf
using SparseArrays
using LinearAlgebra

# ============================================================
# Rosenbrock function
# ============================================================
function rosenbrock(x::AbstractVector)
    s = zero(eltype(x))
    for i in 1:length(x)-1
        s += 100*(x[i+1] - x[i]^2)^2 + (1 - x[i])^2
    end
    return s
end

# ============================================================
# Analytical Gradient
# ============================================================
function rosenbrock_grad(x::Vector{Float64})
    n = length(x)
    g = zeros(n)

    for i in 1:n
        if i > 1
            g[i] += 200*(x[i] - x[i-1]^2)
        end
        if i < n
            g[i] += -400*x[i]*(x[i+1] - x[i]^2) - 2*(1 - x[i])
        end
    end

    return g
end

# ============================================================
# Sparse Hessian
# Structure: tridiagonal
# ============================================================
function rosenbrock_hessian_sparse(x::Vector{Float64})
    n = length(x)

    I = Int[]
    J = Int[]
    V = Float64[]

    for i in 1:n
        # Diagonal
        val = 0.0
        if i > 1
            val += 200
        end
        if i < n
            val += -400*(x[i+1] - 3*x[i]^2) + 2
        end

        push!(I, i); push!(J, i); push!(V, val)

        # Off-diagonal
        if i < n
            val_off = -400*x[i]
            push!(I, i); push!(J, i+1); push!(V, val_off)
            push!(I, i+1); push!(J, i); push!(V, val_off)
        end
    end

    return sparse(I, J, V, n, n)
end

# ============================================================
# Test point and directions
# ============================================================
x0 = [-1.2, 1.0, -0.5, 2.0, -1.0, 0.5]
n = length(x0)

# Direction vectors
du = randn(n)  #du = du/norm(du)
dv = randn(n)  #dv = dv/norm(dv)

println("x0 = ", x0)
println("du = ", du)
println("dv = ", dv)

# ============================================================
# Analytical comp
# ============================================================
g = rosenbrock_grad(x0)
H = rosenbrock_hessian_sparse(x0)

df_du_analytic = dot(g, du)
df_dv_analytic = dot(g, dv)
d2f_dudv_analytic = dot(du, H * dv)

println("\nAnalytical:")
println("Df(x)[du] = ", df_du_analytic)
println("Df(x)[dv] = ", df_dv_analytic)
println("D²f(x)[du,dv] = ", d2f_dudv_analytic)

x0 = [-1.2, 1.0, -0.5, 2.0, -1.0, 0.5]
du = [0.891234491239826, 0.0176907953731731, -0.9406095550935382, 1.0287045021198407, -0.7379432668733455, -0.10988389438701969]
dv = [0.21095092367784252, 1.4144358093195268, -0.9205208828351517, 0.8792073784077122, -0.8480848290545241, -0.8146969900997977]

Analytical:
Df(x)[du] = 5149.092957810119
Df(x)[dv] = 5564.319977755394
D²f(x)[du,dv] = 7973.602995281478


Set up nested duals.  tags are needed. 

In [69]:
using ForwardDiff
# ============================================================
# Nested Dual Setup with Tags
# ============================================================
# Outer tag = :u
# Inner tag = :v
const TagU = ForwardDiff.Tag(:u, Float64)
const TagV = ForwardDiff.Tag(:v, Float64)

# Step 1: build inner duals (v-direction)
x_v = Vector{ForwardDiff.Dual{TagV,Float64,1}}(undef, n)
for i in 1:n
    x_v[i] = ForwardDiff.Dual{TagV}(x0[i], (dv[i],))
end

# Step 2: wrap with outer duals (u-direction)
x_uv = Vector{ForwardDiff.Dual{TagU,typeof(x_v[1]),1}}(undef, n)
for i in 1:n
    x_uv[i] = ForwardDiff.Dual{TagU}(x_v[i], (du[i],))
end

# ============================================================
# Evaluate function on nested duals
# ============================================================
y = rosenbrock(x_uv)

Dual{ForwardDiff.Tag{Symbol, Float64}()}(Dual{ForwardDiff.Tag{Symbol, Float64}()}(3087.7,5564.319977755393),Dual{ForwardDiff.Tag{Symbol, Float64}()}(5149.092957810119,7973.60299528148))

In [70]:
# ============================================================
# Extraction
# ============================================================

# Function value
f_val = ForwardDiff.value(ForwardDiff.value(y))

# First derivatives
df_du = ForwardDiff.value(ForwardDiff.partials(y)[1])        # u outer derivative
df_dv = ForwardDiff.partials(ForwardDiff.value(y))[1]        # v inner derivative

# Mixed second derivative
# Take outer derivative, then extract its inner derivative
outer = ForwardDiff.partials(y)[1]
d2f_dudv = ForwardDiff.partials(outer)[1]

println("\nDual-based extraction:")
println("f(x) = ", f_val)
println("Df(x)[du] = ", df_du)
println("Df(x)[dv] = ", df_dv)
println("D²f(x)[du,dv] = ", d2f_dudv)

# ============================================================
# Consistency checks
# ============================================================
println("\nErrors:")
println("|Df_du error| = ", abs(df_du - df_du_analytic))
println("|Df_dv error| = ", abs(df_dv - df_dv_analytic))
println("|D2f_dudv error| = ", abs(d2f_dudv - d2f_dudv_analytic))


Dual-based extraction:
f(x) = 3087.7
Df(x)[du] = 5149.092957810119
Df(x)[dv] = 5564.319977755393
D²f(x)[du,dv] = 7973.60299528148

Errors:
|Df_du error| = 0.0
|Df_dv error| = 9.094947017729282e-13
|D2f_dudv error| = 1.8189894035458565e-12
